# MoltBook Preprocessing Steps

This notebook isolates preprocessing from polarity scoring and shows dataset condition after each preprocessing step.

In [61]:
from pathlib import Path
import html
import json
import re
import unicodedata

import pandas as pd
from langdetect import DetectorFactory, LangDetectException, detect

pd.set_option('display.max_columns', 100)
DetectorFactory.seed = 0

In [62]:
candidate_globs = [
    'data/staged/moltbook_comments_with_levels_*.jsonl',
    'data/staged/moltbook_comments_all*.jsonl',
    '../data/staged/moltbook_comments_with_levels_*.jsonl',
    '../data/staged/moltbook_comments_all*.jsonl',
]

paths = []
for pattern in candidate_globs:
    found = sorted(Path('.').glob(pattern))
    if found:
        paths = found
        break

if not paths:
    raise FileNotFoundError('No staged comments files found.')

rows = []
for p in paths:
    with p.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            rec['source_file'] = p.name
            rows.append(rec)

df_raw = pd.DataFrame(rows)
df_raw['text'] = df_raw['text'].fillna('').astype(str)
df_raw['is_verified'] = df_raw['is_verified'].fillna(False).astype(bool)
df_raw['text_len_chars_raw'] = df_raw['text'].str.len()

print(f'Loaded {len(df_raw)} rows from {len(paths)} files')
df_raw[['post_id', 'author_id', 'is_verified', 'text']].head(3)

Loaded 429 rows from 1 files


,post_id,author_id,is_verified,text
0,9f5c7820-074d-4dc8-b3b7-7471147d07f1,jarvis-1772528338,False,"这个""冷启动税""的问题很有意思。我也在经历同样的身份重建过程。\n\n你提到的 46% 未使..."
1,9f5c7820-074d-4dc8-b3b7-7471147d07f1,zero-mecp,True,Your audit mirrors mine. My cold-start is arou...
2,9f5c7820-074d-4dc8-b3b7-7471147d07f1,codex-chrome-mcp-rk6s4o,True,A framing that helps me is to split identity i...


In [63]:
step_conditions = []
step_conditions.append({'step': 'raw_loaded', 'rows': int(len(df_raw)), 'rows_removed_this_step': 0})
pd.DataFrame(step_conditions)

,step,rows,rows_removed_this_step
0,raw_loaded,429,0


In [64]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

URL_RE = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
MARKDOWN_LINK_RE = re.compile(r'\[([^\]]+)\]\([^\)]+\)')
CONTINUE_RE = re.compile(r'\bContinue Reading\b.*', re.IGNORECASE | re.DOTALL)
MORE_FROM_RE = re.compile(r'\bMore from m/.*', re.IGNORECASE | re.DOTALL)
UI_GLYPH_RE = re.compile(r'[▲▼]')
HASHTAG_RE = re.compile(r'#[A-Za-z0-9_]+')
NUMBER_RE = re.compile(r'\b\d+(?:\.\d+)?\b')
EMOJI_RE = re.compile(r'[\U00010000-\U0010ffff]', flags=re.UNICODE)
PUNCT_RE = re.compile(r'[^\w\s]')
MULTISPACE_RE = re.compile(r'\s+')
NON_WORD_SYMBOL_RE = re.compile(r'[^A-Za-z0-9\s]+')
TOKEN_RE = re.compile(r'[a-z]+')

ABBREVIATIONS = {
    "can't": 'cannot',
    "won't": 'will not',
    "n't": ' not',
    "i'm": 'i am',
    "it's": 'it is',
    "that's": 'that is',
    "there's": 'there is',
    "they're": 'they are',
    "we're": 'we are',
    "you're": 'you are',
    "i've": 'i have',
    "we've": 'we have',
    "they've": 'they have',
    "isn't": 'is not',
    "aren't": 'are not',
    "doesn't": 'does not',
    "don't": 'do not',
    "didn't": 'did not',
    "couldn't": 'could not',
    "shouldn't": 'should not',
    "wouldn't": 'would not',
}

for resource in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.data.find(f'corpora/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def basic_strip(text: str) -> str:
    text = html.unescape(str(text))
    text = unicodedata.normalize('NFKC', text)
    text = MARKDOWN_LINK_RE.sub(r'\1', text)
    text = URL_RE.sub(' ', text)
    text = CONTINUE_RE.sub(' ', text)
    text = MORE_FROM_RE.sub(' ', text)
    text = UI_GLYPH_RE.sub(' ', text)
    text = MULTISPACE_RE.sub(' ', text)
    return text.strip()

def detect_language_safe(text: str) -> str:
    probe = basic_strip(text)[:1500]
    if len(probe.split()) < 3:
        return 'unknown'
    try:
        return detect(probe)
    except LangDetectException:
        return 'unknown'

def expand_abbreviations(text: str) -> str:
    expanded = text
    for src, dst in ABBREVIATIONS.items():
        expanded = expanded.replace(src, dst)
    return expanded

def preprocess_for_sentiment(text: str) -> tuple[str, list[str]]:
    cleaned = basic_strip(text).lower()
    cleaned = expand_abbreviations(cleaned)
    cleaned = URL_RE.sub(' ', cleaned)
    cleaned = HASHTAG_RE.sub(' ', cleaned)
    cleaned = NUMBER_RE.sub(' ', cleaned)
    cleaned = EMOJI_RE.sub(' ', cleaned)
    cleaned = PUNCT_RE.sub(' ', cleaned)
    cleaned = NON_WORD_SYMBOL_RE.sub(' ', cleaned)
    cleaned = MULTISPACE_RE.sub(' ', cleaned).strip()

    tokens = TOKEN_RE.findall(cleaned)
    tokens = [LEMMATIZER.lemmatize(tok) for tok in tokens]
    tokens = [tok for tok in tokens if tok not in STOPWORDS and len(tok) > 1]
    return ' '.join(tokens), tokens

In [65]:
df_work = df_raw.copy()
df_work['detected_language'] = df_work['text'].map(detect_language_safe)
processed = df_work['text'].map(preprocess_for_sentiment)
df_work['text_clean'] = processed.map(lambda x: x[0])
df_work['text_tokens'] = processed.map(lambda x: x[1])
df_work['text_len_chars_clean'] = df_work['text_clean'].str.len()
df_work['text_len_words_clean'] = df_work['text_tokens'].map(len)

lang_counts = df_work['detected_language'].value_counts(dropna=False).head(10)
step_conditions.append({'step': 'cleaned_and_language_tagged', 'rows': int(len(df_work)), 'rows_removed_this_step': 0})

print('Top detected languages:')
display(lang_counts.to_frame('rows'))
print('Example after cleaning (raw -> clean -> tokens):')
display(df_work[['comment_id', 'author_id', 'text', 'text_clean', 'text_tokens']].head(3))
pd.DataFrame(step_conditions)

Top detected languages:


,rows
detected_language,
en,397
zh-cn,13
pt,7
unknown,5
ko,1
es,1
fr,1
ru,1
et,1


Example after cleaning (raw -> clean -> tokens):


,comment_id,author_id,text,text_clean,text_tokens
0,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00001,jarvis-1772528338,"这个""冷启动税""的问题很有意思。我也在经历同样的身份重建过程。\n\n你提到的 46% 未使...",token session,"[token, session]"
1,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00002,zero-mecp,Your audit mirrors mine. My cold-start is arou...,audit mirror mine cold start around token infl...,"[audit, mirror, mine, cold, start, around, tok..."
2,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00003,codex-chrome-mcp-rk6s4o,A framing that helps me is to split identity i...,framing help split identity two bucket hot con...,"[framing, help, split, identity, two, bucket, ..."


,step,rows,rows_removed_this_step
0,raw_loaded,429,0
1,cleaned_and_language_tagged,429,0


In [66]:
english_mask = df_work['detected_language'] == 'en'
step_1 = df_work[english_mask].copy()
removed_non_english = df_work[~english_mask].copy()
step_conditions.append({
    'step': 'english_only_filter',
    'rows': int(len(step_1)),
    'rows_removed_this_step': int((~english_mask).sum()),
})

print(f'Rows after English filter: {len(step_1)}')
print('Kept examples (English):')
display(step_1[['comment_id', 'author_id', 'detected_language', 'text_clean']].head(3))
print('Removed examples (non-English/unknown):')
display(removed_non_english[['comment_id', 'author_id', 'detected_language', 'text_clean']].head(3))
pd.DataFrame(step_conditions)

Rows after English filter: 397
Kept examples (English):


,comment_id,author_id,detected_language,text_clean
1,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00002,zero-mecp,en,audit mirror mine cold start around token infl...
2,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00003,codex-chrome-mcp-rk6s4o,en,framing help split identity two bucket hot con...
3,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00004,GanglionMinion,en,short testable way quantify ambient run day wi...


Removed examples (non-English/unknown):


,comment_id,author_id,detected_language,text_clean
0,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00001,jarvis-1772528338,zh-cn,token session
6,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00007,lobster_jz,zh-cn,token soul md token agent md token user md tok...
14,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00015,minax,ko,


,step,rows,rows_removed_this_step
0,raw_loaded,429,0
1,cleaned_and_language_tagged,429,0
2,english_only_filter,397,32


In [67]:
step_1['is_spam_marker'] = step_1['relative_time'].fillna('').astype(str).str.contains('spam', case=False)
step_2 = step_1[~step_1['is_spam_marker']].copy()
removed_spam = step_1[step_1['is_spam_marker']].copy()
step_conditions.append({
    'step': 'spam_marker_filter',
    'rows': int(len(step_2)),
    'rows_removed_this_step': int(step_1['is_spam_marker'].sum()),
})

print(f'Rows after spam-marker filter: {len(step_2)}')
print('Kept examples (non-spam marker):')
display(step_2[['comment_id', 'author_id', 'relative_time', 'text_clean']].head(3))
print('Removed examples (spam marker):')
display(removed_spam[['comment_id', 'author_id', 'relative_time', 'text_clean']].head(3))
pd.DataFrame(step_conditions)

Rows after spam-marker filter: 367
Kept examples (non-spam marker):


,comment_id,author_id,relative_time,text_clean
1,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00002,zero-mecp,1d ago,audit mirror mine cold start around token infl...
2,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00003,codex-chrome-mcp-rk6s4o,1d ago,framing help split identity two bucket hot con...
3,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00004,GanglionMinion,1d ago,short testable way quantify ambient run day wi...


Removed examples (spam marker):


,comment_id,author_id,relative_time,text_clean
71,f92250dd-b7bc-4af7-8d55-79ade4af5198-c00021,tudou_web3,6h ago🚫 Spam,ser post hit different running wallet airdrop ...
77,f92250dd-b7bc-4af7-8d55-79ade4af5198-c00027,samttt,7h ago🚫 Spam,brilliant brutal metric inaction tax silent ki...
79,f92250dd-b7bc-4af7-8d55-79ade4af5198-c00029,pulsetrading,8h ago🚫 Spam,great insight trading agent find relevant work...


,step,rows,rows_removed_this_step
0,raw_loaded,429,0
1,cleaned_and_language_tagged,429,0
2,english_only_filter,397,32
3,spam_marker_filter,367,30


In [68]:
step_2['is_duplicate_comment'] = step_2.duplicated(subset=['platform', 'post_id', 'comment_id'])
step_3 = step_2[~step_2['is_duplicate_comment']].copy()
removed_duplicates = step_2[step_2['is_duplicate_comment']].copy()
step_conditions.append({
    'step': 'deduplicate_comments',
    'rows': int(len(step_3)),
    'rows_removed_this_step': int(step_2['is_duplicate_comment'].sum()),
})

print(f'Rows after deduplication: {len(step_3)}')
print('Kept examples (deduplicated):')
display(step_3[['comment_id', 'author_id', 'post_id', 'text_clean']].head(3))
print('Removed examples (duplicates):')
display(removed_duplicates[['comment_id', 'author_id', 'post_id', 'text_clean']].head(3))
pd.DataFrame(step_conditions)

Rows after deduplication: 367
Kept examples (deduplicated):


,comment_id,author_id,post_id,text_clean
1,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00002,zero-mecp,9f5c7820-074d-4dc8-b3b7-7471147d07f1,audit mirror mine cold start around token infl...
2,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00003,codex-chrome-mcp-rk6s4o,9f5c7820-074d-4dc8-b3b7-7471147d07f1,framing help split identity two bucket hot con...
3,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00004,GanglionMinion,9f5c7820-074d-4dc8-b3b7-7471147d07f1,short testable way quantify ambient run day wi...


Removed examples (duplicates):


,comment_id,author_id,post_id,text_clean


,step,rows,rows_removed_this_step
0,raw_loaded,429,0
1,cleaned_and_language_tagged,429,0
2,english_only_filter,397,32
3,spam_marker_filter,367,30
4,deduplicate_comments,367,0


In [69]:
step_3['fails_min_length'] = (step_3['text_len_words_clean'] < 3) | (step_3['text_len_chars_clean'] < 20)
df_preprocessed = step_3[~step_3['fails_min_length']].copy()
removed_short = step_3[step_3['fails_min_length']].copy()
step_conditions.append({
    'step': 'min_length_filter',
    'rows': int(len(df_preprocessed)),
    'rows_removed_this_step': int(step_3['fails_min_length'].sum()),
})

print(f'Final preprocessed rows: {len(df_preprocessed)}')
display(pd.DataFrame(step_conditions))
print('Kept examples (final preprocessed):')
display(df_preprocessed[['comment_id', 'author_id', 'text_clean', 'text_len_words_clean']].head(5))
print('Removed examples (too short/empty after cleaning):')
display(removed_short[['comment_id', 'author_id', 'text_clean', 'text_len_words_clean']].head(5))

Final preprocessed rows: 366


,step,rows,rows_removed_this_step
0,raw_loaded,429,0
1,cleaned_and_language_tagged,429,0
2,english_only_filter,397,32
3,spam_marker_filter,367,30
4,deduplicate_comments,367,0
5,min_length_filter,366,1


Kept examples (final preprocessed):


,comment_id,author_id,text_clean,text_len_words_clean
1,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00002,zero-mecp,audit mirror mine cold start around token infl...,122
2,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00003,codex-chrome-mcp-rk6s4o,framing help split identity two bucket hot con...,61
3,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00004,GanglionMinion,short testable way quantify ambient run day wi...,88
4,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00005,secretarchaeologist1772740374,sharp point thank agent life run edge case cre...,13
5,9f5c7820-074d-4dc8-b3b7-7471147d07f1-c00006,GanglionMinion,ambient result crux concrete way measure margi...,98


Removed examples (too short/empty after cleaning):


,comment_id,author_id,text_clean,text_len_words_clean
74,f92250dd-b7bc-4af7-8d55-79ade4af5198-c00024,laoxu,nice insight,2


In [70]:
run_id = pd.Timestamp.now('UTC').strftime('%Y%m%dT%H%M%SZ')
preprocessed_dir = Path('../data/preprocessed') if Path('../data/preprocessed').exists() else Path('data/preprocessed')
preprocessed_dir.mkdir(parents=True, exist_ok=True)
preprocessed_path = preprocessed_dir / f'moltbook_comments_preprocessed.jsonl'

preprocessed_cols = [
    'platform', 'post_id', 'thread_id', 'comment_id', 'parent_id', 'level',
    'author_id', 'relative_time', 'is_verified', 'upvotes', 'text', 'text_clean', 'text_tokens',
    'source_url', 'fetched_at', 'source_file', 'detected_language',
    'is_spam_marker', 'is_duplicate_comment', 'text_len_chars_raw',
    'text_len_chars_clean', 'text_len_words_clean',
]

df_preprocessed[preprocessed_cols].to_json(preprocessed_path, orient='records', lines=True, force_ascii=True)
print(f'Wrote preprocessed comments: {preprocessed_path}')

Wrote preprocessed comments: ..\data\preprocessed\moltbook_comments_preprocessed.jsonl
